## INSTALAÇÃO DAS BIBLIOTECAS E IMPORTAÇÕES DO PROJETO: ##

In [1]:
pip install -r requirements.txt

  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached cffi-2.1.1-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-win_amd64.whl.metadata (47 kB)
  Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl.metadata (4.4 kB)
  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached google_api_core-2.25.2-py3-none-any.whl.metadata (3.0 kB)
  Using cached google_api_python_client-2.201.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached google_auth-2.62.1-py3-none-any.whl.metadata (6.0 kB)
  Using cached google_auth_httplib2-0.4.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached grpcio-1.85.1-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using ca


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
import json 
import requests
import pandas as pd
import time

c:\Users\francesco.antony\Documents\local\Feedback Colaborativo\Projeto Final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\francesco.antony\AppData\Local\Temp\ipykernel_24108\4154645045.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### MODELO DE IA UTILIZADO E VARIÁVEIS DE AMBIENTE: ###

In [3]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY") # Sua API do gemini
genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-3.1-flash-lite") # Modelo escolhido

WEBHOOK_N8N_TEST_URL = os.getenv("WEBHOOK_N8N_TEST_URL") # Teste local
WEBHOOK_N8N_URL = os.getenv("WEBHOOK_N8N_URL") # Ambiente de produção

In [4]:
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models

### FUNÇÃO PARA ANALISAR FEEDBACK VIA URL WEBHOOK DO N8N: ###

In [5]:
def analisar_e_enviar_feedback(nome, email, departamento, feedback_texto):

    print(f"🔄 Processando feedback de {nome} com o Gemini...")
    prompt = f"""
    Você é um especialista em Recursos Humanos. Analise o feedback enviado pelo funcionário e retorne EXCLUSIVAMENTE em formato JSON puro, sem blocos de código markdown (sem ```json), contendo exatamente estas chaves:
    - resumo (string curta de até 2 frases)
    - sentimento (Positivo, Neutro ou Negativo)
    - prioridade (Baixa, Média, Alta ou Urgente)
    - pontos_positivos (lista de strings)
    - pontos_criticos (lista de strings)
    - recomendacoes_gestao (lista de strings)

    Feedback do colaborador: "{feedback_texto}"
    """
    try:
        response = model.generate_content(prompt)

        texto_limpo = response.text.replace("```json", "").replace("```", "").strip()
        dados_ia = json.loads(texto_limpo)

        payload = {
            "nome": nome,
            "email": email,
            "departamento": departamento,
            "feedback_bruto": feedback_texto,
            "resumo": dados_ia.get("resumo"),
            "sentimento": dados_ia.get("sentimento"),
            "prioridade": dados_ia.get("prioridade"),
            "pontos_positivos": dados_ia.get("pontos_positivos"),
            "pontos_criticos": dados_ia.get("pontos_criticos"),
            "recomendacoes_gestao": dados_ia.get("recomendacoes_gestao"),
        }

        print("🚀 Enviando dados processados para o n8n local...")

        resposta_n8n = requests.post(WEBHOOK_N8N_TEST_URL, json=payload)

        if resposta_n8n.status_code == 200:
            print(f"✅ Sucesso! O n8n recebeu os dados e disparou a automação.")
        else:
            print(f"⚠️ O n8n respondeu com o código de erro: {resposta_n8n.status_code}")


    except Exception as e:
        print(f"❌ Erro durante o processo: {e}")


### TESTE COM PARÂMETROS: (nome, email, departamento, feedback_texto) ###

In [6]:
analisar_e_enviar_feedback(
    nome="Teste",
    email="teste@exemplo.com",
    departamento="Teste",
    feedback_texto="É apenas um teste. Nada demais."
)

🔄 Processando feedback de Teste com o Gemini...
🚀 Enviando dados processados para o n8n local...
✅ Sucesso! O n8n recebeu os dados e disparou a automação.


### ENVIAR E CARREGAR DADOS DA PLANILHA VIA URL CSV (WEBHOOK GOOGLE SHEETS) PARA O N8N: ###

In [8]:
def analisar_e_enviar_feedback(feedback_item):

    nome = feedback_item.get("nome")
    email = feedback_item.get("email")
    departamento = feedback_item.get("departamento")
    feedback_texto = feedback_item.get("feedback_texto")
    status = feedback_item.get("status")

    if not status or str(status).lower() == 'nan':
        status = "Pendente"
    else:
        status = status

    linha_planilha = feedback_item.get("linha_planilha")

    print(f"🔄 Processando feedback de {nome} com o Gemini...")
    prompt = f"""
    Você é um especialista em Recursos Humanos. Analise o feedback enviado pelo funcionário e retorne EXCLUSIVAMENTE em formato JSON puro, sem blocos de código markdown (sem ```json), contendo exatamente estas chaves:
    - resumo (string curta de até 2 frases)
    - sentimento (Positivo, Neutro ou Negativo)
    - prioridade (Baixa, Média, Alta ou Urgente)
    - pontos_positivos (lista de strings)
    - pontos_criticos (lista de strings)
    - recomendacoes_gestao (lista de strings)

    Feedback do colaborador: "{feedback_texto}"
    """
    try:
        response = model.generate_content(prompt)

        texto_limpo = response.text.replace("```json", "").replace("```", "").strip()
        dados_ia = json.loads(texto_limpo)

        payload = {
            "nome": nome,
            "email": email,
            "departamento": departamento,
            "feedback_bruto": feedback_texto,
            "resumo": dados_ia.get("resumo"),
            "sentimento": dados_ia.get("sentimento"),
            "prioridade": dados_ia.get("prioridade"),
            "pontos_positivos": dados_ia.get("pontos_positivos"),
            "pontos_criticos": dados_ia.get("pontos_criticos"),
            "recomendacoes_gestao": dados_ia.get("recomendacoes_gestao"),
            "linha_planilha": linha_planilha,
            "status": status
        }

        return payload

    except Exception as e:
        print(f"❌ Erro durante o processo: {e}")


In [9]:
load_dotenv()

URL_CSV = os.getenv("URL_CSV")

df = pd.read_csv(URL_CSV)

total_linhas = len(df) 
print(f"Total de linhas encontradas para processamento: {total_linhas}")

if total_linhas > 0:
    lote_para_n8n = []

    print("\n📦 Agrupando todas as linhas para envio em lote...")
    
    for index, row in df.iterrows():
        status = str(row.get('Status:', '')).strip().lower()
        nome = row.get('Nome:', 'Não encontrado')
        email = row.get('E-mail:', 'Não encontrado')

        if status.startswith('processado'):
            print(f"⏭️ [{index + 1}/{total_linhas}] Pulando {nome} — Já consta como Processado.")
            continue

        print(f"🔄 [{index + 1}/{total_linhas}] Analisando novo feedback de: {nome}")

        feedback_item = {
            "nome": nome,
            "email": row.get('E-mail:', 'Não encontrado'),
            "departamento": row.get('Departamento:', 'Não encontrado'),
            "feedback_texto": row.get('Deixe aqui seu Feedback:', 'Não encontrado'),
            "linha_planilha": index + 2 
        }

        resultado = analisar_e_enviar_feedback(feedback_item)
        if resultado:
            lote_para_n8n.append(resultado)
            
        time.sleep(1) 

    if len(lote_para_n8n) > 0:
        print(f"\n🚀 Enviando lote com {len(lote_para_n8n)} novos feedbacks processados para o n8n...")
        
        try:
            resposta_n8n = requests.post(WEBHOOK_N8N_TEST_URL, json=lote_para_n8n)

            if resposta_n8n.status_code == 200:
                print(f"✅ Sucesso! O n8n recebeu os dados e disparou a automação.")
            else:
                print(f"⚠️ O n8n respondeu com o código de erro: {resposta_n8n.status_code}")
                
        except Exception as e:
            print(f"❌ Erro ao conectar com o n8n: {e}")
    else:
        print("\n😎 Tudo atualizado! Nenhuma linha nova pendente de processamento hoje.")

else:
    print("Nenhum dado encontrado na planilha.")

Total de linhas encontradas para processamento: 8

📦 Agrupando todas as linhas para envio em lote...
⏭️ [1/8] Pulando Carlos Souza — Já consta como Processado.
⏭️ [2/8] Pulando João Raivinho — Já consta como Processado.
⏭️ [3/8] Pulando Francesco — Já consta como Processado.
⏭️ [4/8] Pulando Ana Claudia Costa dos santos — Já consta como Processado.
⏭️ [5/8] Pulando Ricardo Almeida — Já consta como Processado.
⏭️ [6/8] Pulando Francisco Igor de Oliveira Lima  — Já consta como Processado.
⏭️ [7/8] Pulando Peterson — Já consta como Processado.
🔄 [8/8] Analisando novo feedback de: Francinaldo Rei
🔄 Processando feedback de Francinaldo Rei com o Gemini...

🚀 Enviando lote com 1 novos feedbacks processados para o n8n...
✅ Sucesso! O n8n recebeu os dados e disparou a automação.


In [65]:
load_dotenv()

URL_CSV = os.getenv("URL_CSV")

df = pd.read_csv(URL_CSV)

print(f"Total de linhas encontradas: {len(df)}")

if len(df) > 0:

    ultima_resposta = df.iloc[-1]

    nome = ultima_resposta.get('Nome:', 'Não encontrado')
    email = ultima_resposta.get('E-mail:', 'Não encontrado')
    departamento = ultima_resposta.get('Departamento:', 'Não encontrado')
    feedback_texto = ultima_resposta.get('Deixe aqui seu Feedback:', 'Não encontrado')

    print("\nÚltimo feedback recebido:")
    print(f"Nome: {nome}")
    print(f"E-mail: {email}")
    print(f"Departamento: {departamento}")
    print(f"Feedback: {feedback_texto}")

else:
    print("\nA planilha ainda está vazia no link publicado. Confirme se selecionou a aba 'Form_Responses' ao publicar.")

Total de linhas encontradas: 9

Último feedback recebido:
Nome: teste
E-mail: teste@gmail.com
Departamento: teste
Feedback: teste
